In [8]:
import pandas as pd
import mysql.connector


try:
    conn = mysql.connector.connect(
        host="localhost",
        user="root",              
        password="GuNa08", 
        database="amazon_sales_db" 
    )
    print("Database Connection Success! 🎉")
    
    
    query = "SELECT * FROM amazon_products_sales_data_uncleaned"
    df = pd.read_sql(query, conn)
    print("Data successfully loaded into Python! 🚀")
    
except Exception as e:
    print("Opps! Connection Error: ", e)


df.head(3)

Database Connection Success! 🎉
Data successfully loaded into Python! 🚀


C:\Users\GUNAVATHI\AppData\Local\Temp\ipykernel_16060\1672580002.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,title,rating,number_of_reviews,bought_in_last_month,current/discounted_price,price_on_variant,listed_price,is_best_seller,is_sponsored,is_couponed,buy_box_availability,delivery_details,sustainability_badges,image_url,product_url,collected_at
0,BOYA BOYALINK 2 Wireless Lavalier Microphone f...,4.6 out of 5 stars,375,300+ bought in past month,89.68,basic variant price: 2.4GHz,$159.00,No Badge,Sponsored,Save 15% with coupon,Add to cart,"Delivery Mon, Sep 1",Carbon impact,https://m.media-amazon.com/images/I/71pAqiVEs3...,/sspa/click?ie=UTF8&spc=MTo4NzEzNDY2NTQ5NDYxND...,2025-08-21 11:14:29
1,"LISEN USB C to Lightning Cable, 240W 4 in 1 Ch...",4.3 out of 5 stars,"2,457",6K+ bought in past month,9.99,basic variant price: nan,$15.99,No Badge,Sponsored,No Coupon,Add to cart,"Delivery Fri, Aug 29",,https://m.media-amazon.com/images/I/61nbF6aVIP...,/sspa/click?ie=UTF8&spc=MTo4NzEzNDY2NTQ5NDYxND...,2025-08-21 11:14:29
2,"DJI Mic 2 (2 TX + 1 RX + Charging Case), Wirel...",4.6 out of 5 stars,"3,044",2K+ bought in past month,314.00,basic variant price: nan,$349.00,No Badge,Sponsored,No Coupon,Add to cart,"Delivery Mon, Sep 1",,https://m.media-amazon.com/images/I/61h78MEXoj...,/sspa/click?ie=UTF8&spc=MTo4NzEzNDY2NTQ5NDYxND...,2025-08-21 11:14:29


In [9]:
df['rating_cleaned'] = df['rating'].str.split(' ').str[0]
df['rating_cleaned'] = pd.to_numeric(df['rating_cleaned'], errors='coerce')

df['bought_cleaned'] = df['bought_in_last_month'].str.extract(r'(\d+K\+|\d+\+)')
df['bought_cleaned'] = df['bought_cleaned'].str.replace('+', '', regex=False)

def handle_k_values(val):
    if pd.isna(val):
        return 0
    if 'K' in str(val):
        return float(val.replace('K', '')) * 1000
    return float(val)

df['bought_cleaned'] = df['bought_cleaned'].apply(handle_k_values)

df['price_cleaned'] = pd.to_numeric(df['current/discounted_price'], errors='coerce')

df['estimated_revenue'] = df['price_cleaned'] * df['bought_cleaned']

df['number_of_reviews'] = pd.to_numeric(df['number_of_reviews'], errors='coerce')

print("Cleaning processing complete! 🔥")

df[['title', 'rating_cleaned', 'bought_cleaned', 'price_cleaned', 'estimated_revenue']].head()

Cleaning processing complete! 🔥


,title,rating_cleaned,bought_cleaned,price_cleaned,estimated_revenue
0,BOYA BOYALINK 2 Wireless Lavalier Microphone f...,4.6,300.0,89.68,26904.0
1,"LISEN USB C to Lightning Cable, 240W 4 in 1 Ch...",4.3,6000.0,9.99,59940.0
2,"DJI Mic 2 (2 TX + 1 RX + Charging Case), Wirel...",4.6,2000.0,314.00,628000.0
3,"Apple AirPods Pro 2 Wireless Earbuds, Active N...",4.6,10000.0,NaN,NaN
4,Apple AirTag 4 Pack. Keep Track of and find Yo...,4.8,10000.0,NaN,NaN


In [10]:
df.to_excel("amazon_sales_cleaned_final.xlsx", index=False)
